# 02 Data preparation — corrected copy

This notebook is the maintained, runnable preparation walkthrough. The original `02_data_preparation.ipynb` is preserved exactly for side-by-side comparison. This copy uses the maintained `src` modules, frozen manifest, and canonical paths; it does not rewrite documentation or source modules.

## Source, target, and frozen assignments

The canonical source is `data/raw/heart_2022_with_nans.zip`. Source row order defines `row_id`. Exact matches across all original fields share `duplicate_group` and stay in one split. `heart_disease` is 1 if HadHeartAttack or HadAngina is Yes, 0 only if both are No, and otherwise unknown. Unknown targets are recorded and excluded. The two target-source fields are excluded from predictor exports. The frozen manifest is validated, never regenerated by this notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

candidates = (Path.cwd(), *Path.cwd().parents)
ROOT = next((p for p in candidates if (p / "src" / "prepare_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository root or its notebooks directory")
sys.path.insert(0, str(ROOT))
from src.prepare_data import prepare_data, SOURCE, MANIFEST_DIR, OUTPUT_DIR, AUDIT_DIR
print("Repository:", ROOT)
print("Source:", SOURCE)
print("Observed splits:", OUTPUT_DIR)
print("Frozen manifest:", MANIFEST_DIR / "split_manifest.csv")

In [ ]:
splits = prepare_data(check_seed_42=True)
assert {name: len(frame) for name, frame in splits.items()} == {"train": 308080, "validation": 66016, "test": 66015}
assert all(len(frame.columns) == 41 for frame in splits.values())
assert all("HadHeartAttack" not in frame and "HadAngina" not in frame for frame in splits.values())
assert all(frame.drop(columns=["row_id", "duplicate_group", "heart_disease"]).isna().any().any() for frame in splits.values())
display(pd.read_csv(AUDIT_DIR / "split_summary.csv"))

## Observed handoff and audits

`data/processed/observed/training.csv` is Member 2’s input for EDA and observed-value associations. Validation is reserved for model decisions and test for final evaluation. All 38 original predictor values and missing positions remain intact. `heart_disease` is saved as the target but separated as `y` for modeling; `row_id` and `duplicate_group` are metadata, not predictors. Training missingness and numeric ranges are audited below. Flags prompt review and never delete rows. SleepHours uses the dictionary’s 1–24 range.

In [ ]:
training = splits["train"]
X = training.drop(columns=["heart_disease", "row_id", "duplicate_group"])
y = training["heart_disease"]
print("Training rows with any missing predictor:", int(X.isna().any(axis=1).sum()))
print("Training rows with missing BMI:", int(X.BMI.isna().sum()))
display(pd.read_csv(AUDIT_DIR / "training_missing_summary.csv"))
display(pd.read_csv(AUDIT_DIR / "training_range_flags.csv"))

## Modeling handoff

`src.preprocessing.make_preprocessor(training)` returns a new unfitted transformer. Fit it inside an estimator `Pipeline` in each cross-validation training fold. Use `data/processed/observed/` for EDA and cross-validation, with missing values preserved. Optional `--write-imputed` copies in `data/processed/imputed/` use training-fitted numeric medians and a categorical Missing label. Those copies are not inputs for observed-data associations or cross-validation. The older imputed archive is historical.